# Model Training — Insurance Premium Prediction

**Objective**: Train and compare regression models to predict medical insurance
`charges` from customer attributes, then select, tune, and persist the best model.

This notebook builds directly on the feature pipeline defined in
`src/features/build_features.py` (encoding + smoker×BMI interaction + split), so the
training data here is guaranteed identical to what production will use.

**Workflow**
1. Load raw data and apply the feature pipeline
2. Establish a naive baseline (a model must beat this to be worth anything)
3. Cross-validate a spread of models (linear → regularized → tree ensembles)
4. Tune the best performer with randomized search
5. Evaluate the final model on the held-out test set
6. Inspect residuals and feature importance
7. Log the run to MLflow and persist the model artifact


## 1. Setup & Imports

We import from `src/features` so the notebook and the production pipeline share a
single source of truth — no re-implementing feature logic here.

In [ ]:
# Make the project root importable so `from src.features...` works
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor

from src.features.build_features import build_features, split_data

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

print("Project root:", PROJECT_ROOT)

## 2. Load Data & Build Features

We load the raw CSV and pass it straight through `build_features()`. Keeping this
call here (rather than reading a pre-processed file) means any change to the pipeline
is reflected automatically the next time this notebook runs.

In [ ]:
raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "insurance.csv")
print(f"Raw shape: {raw.shape}")
raw.head()

In [ ]:
df = build_features(raw)
print(f"Engineered shape: {df.shape}")
print("Columns:", list(df.columns))
df.head()

### Train/test split

Using the project's `split_data()` helper (80/20, `random_state=42`) so the split is
reproducible and consistent with the rest of the codebase. We hold the test set out
completely — it is touched only once, at the very end.

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, target_col="charges",
                                              test_size=0.2, random_state=RANDOM_STATE)

FEATURE_COLS = list(X_train.columns)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print("Features:", FEATURE_COLS)

The target `charges` is strongly right-skewed (a few very expensive claims). This
matters for model choice and for reading RMSE, so let's confirm it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y_train, bins=40, kde=True, ax=axes[0])
axes[0].set_title("charges (raw)")
sns.histplot(np.log1p(y_train), bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("charges (log1p)")
plt.tight_layout(); plt.show()

print(f"Skew (raw):   {y_train.skew():.3f}")
print(f"Skew (log1p): {np.log1p(y_train).skew():.3f}")

## 3. Baseline

Before any real model, we fit a `DummyRegressor` that always predicts the mean of the
training target. Every model below must comfortably beat this — it's the floor that
tells us whether the features carry any signal at all.

In [ ]:
def regression_report(y_true, y_pred, label=""):
    """Return a dict of the three metrics we track everywhere."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    if label:
        print(f"{label:<22} RMSE={rmse:10.2f}  MAE={mae:10.2f}  R2={r2:6.3f}")
    return {"rmse": rmse, "mae": mae, "r2": r2}


dummy = DummyRegressor(strategy="mean").fit(X_train, y_train)
_ = regression_report(y_test, dummy.predict(X_test), "Baseline (mean)")

## 4. Model Comparison (Cross-Validated)

We compare a spread of model families using 5-fold CV on the **training set only**.
Linear models get a `StandardScaler` (they're scale-sensitive); tree ensembles don't
need it. We score on RMSE, MAE, and R² and average across folds.

CV on train keeps the test set pristine — model selection must never peek at it.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def scaled(model):
    """Wrap a scale-sensitive model with StandardScaler."""
    return Pipeline([("scaler", StandardScaler()), ("model", model)])

candidates = {
    "LinearRegression": scaled(LinearRegression()),
    "Ridge":            scaled(Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    "Lasso":            scaled(Lasso(alpha=1.0, random_state=RANDOM_STATE)),
    "RandomForest":     RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost":          XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=3,
                                     subsample=0.9, colsample_bytree=0.9,
                                     random_state=RANDOM_STATE, n_jobs=-1),
}

scoring = {"rmse": "neg_root_mean_squared_error",
           "mae": "neg_mean_absolute_error",
           "r2": "r2"}

rows = []
for name, est in candidates.items():
    res = cross_validate(est, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        "model": name,
        "cv_rmse": -res["test_rmse"].mean(),
        "cv_rmse_std": res["test_rmse"].std(),
        "cv_mae": -res["test_mae"].mean(),
        "cv_r2": res["test_r2"].mean(),
    })

cv_results = (pd.DataFrame(rows)
              .sort_values("cv_rmse")
              .reset_index(drop=True))
cv_results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(cv_results["model"], cv_results["cv_rmse"],
        xerr=cv_results["cv_rmse_std"], color="steelblue", alpha=0.85)
ax.invert_yaxis()
ax.set_xlabel("CV RMSE (lower is better)")
ax.set_title("5-fold cross-validated RMSE by model")
plt.tight_layout(); plt.show()

best_name = cv_results.iloc[0]["model"]
print(f"Best model by CV RMSE: {best_name}")

## 5. Hyperparameter Tuning

We tune the top performer with `RandomizedSearchCV` (cheaper than an exhaustive grid
and usually finds a comparable optimum). The search runs 5-fold CV internally, still
only on the training data.

The grid below targets the two tree ensembles, which are the usual winners on this
dataset. If a linear model wins instead, we skip tuning — there is little to tune.

In [ ]:
param_grids = {
    "XGBoost": {
        "n_estimators": [300, 400, 600, 800],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "max_depth": [2, 3, 4, 5],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "min_child_weight": [1, 3, 5],
    },
    "RandomForest": {
        "n_estimators": [200, 400, 600, 800],
        "max_depth": [None, 4, 6, 8, 12],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 1.0],
    },
    "GradientBoosting": {
        "n_estimators": [200, 300, 500],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "max_depth": [2, 3, 4],
        "subsample": [0.8, 0.9, 1.0],
    },
}

if best_name in param_grids:
    base = candidates[best_name]
    search = RandomizedSearchCV(
        base, param_grids[best_name], n_iter=40, cv=cv,
        scoring="neg_root_mean_squared_error", n_jobs=-1,
        random_state=RANDOM_STATE, refit=True, verbose=1,
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    print("Best params:", search.best_params_)
    print(f"Best CV RMSE: {-search.best_score_:.2f}")
else:
    best_model = candidates[best_name].fit(X_train, y_train)
    print(f"{best_name} needs no tuning; fitted with defaults.")

## 6. Final Evaluation on the Held-Out Test Set

This is the **only** time we use the test set. These numbers are our honest estimate
of how the model will perform on unseen customers.

In [ ]:
best_model.fit(X_train, y_train)   # refit on full training data
y_pred = best_model.predict(X_test)

print(f"Final model: {best_name}\n")
regression_report(y_train, best_model.predict(X_train), "Train")
test_metrics = regression_report(y_test, y_pred, "Test")

gap = r2_score(y_train, best_model.predict(X_train)) - test_metrics["r2"]
print(f"\nTrain–test R2 gap: {gap:.3f}  (large gap => overfitting)")

## 7. Diagnostics

**Predicted vs. actual** should hug the diagonal. **Residuals vs. predicted** should
be a shapeless cloud around zero — any funnel or curve signals the model is missing
structure (often the log-skew of `charges`).

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, y_pred, alpha=0.5, edgecolor="k", linewidth=0.3)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--", lw=1.5)
axes[0].set(xlabel="Actual charges", ylabel="Predicted charges",
            title="Predicted vs. Actual")

axes[1].scatter(y_pred, residuals, alpha=0.5, edgecolor="k", linewidth=0.3)
axes[1].axhline(0, color="r", ls="--", lw=1.5)
axes[1].set(xlabel="Predicted charges", ylabel="Residual (actual − pred)",
            title="Residuals vs. Predicted")
plt.tight_layout(); plt.show()

### Feature importance

For tree models we read `feature_importances_`; for linear models we use the absolute
standardized coefficients. This confirms whether the model relies on the drivers EDA
flagged — smoker status, the smoker×BMI interaction, and age.

In [ ]:
def get_importances(model, feature_names):
    """Extract importances from a fitted estimator or pipeline."""
    est = model.named_steps["model"] if isinstance(model, Pipeline) else model
    if hasattr(est, "feature_importances_"):
        return pd.Series(est.feature_importances_, index=feature_names)
    if hasattr(est, "coef_"):
        return pd.Series(np.abs(est.coef_), index=feature_names)
    return None

imp = get_importances(best_model, FEATURE_COLS)
if imp is not None:
    imp = imp.sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(imp.index, imp.values, color="seagreen", alpha=0.85)
    ax.set_title(f"Feature importance — {best_name}")
    plt.tight_layout(); plt.show()
    display(imp.sort_values(ascending=False).to_frame("importance"))
else:
    print("Model does not expose importances.")

## 8. Experiment Tracking with MLflow

We log the winning run — parameters, the three test metrics, and the model artifact —
so results are reproducible and comparable across future experiments. Runs are written
to a local `mlruns/` folder; view them with `mlflow ui` from the project root.

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri((PROJECT_ROOT / "mlruns").as_uri())
mlflow.set_experiment("insurance-premium-prediction")

with mlflow.start_run(run_name=f"{best_name}-tuned"):
    mlflow.log_param("model_family", best_name)
    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", RANDOM_STATE)

    est = best_model.named_steps["model"] if isinstance(best_model, Pipeline) else best_model
    for k, v in est.get_params().items():
        mlflow.log_param(f"hp_{k}", v)

    mlflow.log_metric("test_rmse", test_metrics["rmse"])
    mlflow.log_metric("test_mae", test_metrics["mae"])
    mlflow.log_metric("test_r2", test_metrics["r2"])

    mlflow.sklearn.log_model(best_model, name="model")
    print("Logged run to MLflow. Launch the UI with:  mlflow ui")

## 9. Persist the Final Model

We save the fitted model together with the exact feature schema it expects. Storing
`feature_names` alongside the model lets the serving layer validate and order incoming
columns — a common source of silent train/serve skew if left implicit.

In [ ]:
import joblib
from datetime import datetime, timezone

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

artifact = {
    "model": best_model,
    "feature_names": FEATURE_COLS,
    "target": "charges",
    "model_family": best_name,
    "metrics": test_metrics,
    "trained_at": datetime.now(timezone.utc).isoformat(),
}

out_path = MODELS_DIR / "insurance_model.joblib"
joblib.dump(artifact, out_path)
print(f"Saved model → {out_path}")

# Sanity-check the round-trip
loaded = joblib.load(out_path)
sample = X_test.iloc[[0]][loaded["feature_names"]]
print("Reloaded model prediction on one row:", loaded["model"].predict(sample)[0])

---

### Summary & next steps

- Trained and cross-validated six model families against a mean baseline.
- Tuned the best performer and evaluated it once on the held-out test set.
- Logged the run to MLflow and persisted the model with its feature schema.

**Next in the MLOps pipeline**
- Promote the notebook logic into `src/models/train.py` for a reproducible CLI run.
- Wire the saved artifact into `src/api/` for serving.
- Add drift/performance checks in `src/monitoring/`.
